# Deliverable 3 · lote real en diez bloques

50 preguntas × seis condiciones = 300 respuestas. Cada bloque tiene **30 respuestas** (cinco preguntas en las seis condiciones). No ejecuta el piloto.

1. En una sesión nueva T4, ejecutar preparación, restauración y carga (secciones 1–3).
2. Ejecutar **un bloque** en la última celda. Restaura el respaldo verificado de 39 respuestas cuando haga falta y respeta resultados posteriores ya presentes en la sesión.
3. Descargar y auditar el ZIP al terminar. Cada seis respuestas nuevas se solicita además un respaldo parcial. Comprobar que realmente llegó a Descargas.
4. Para el siguiente bloque en la misma sesión, ejecutar **solo la última celda**; no volver a cargar el modelo. Se detiene en 60, 90, …, 300 según lo ya guardado.

El primer bloque ya está completo. Partiendo del respaldo de 39, se generan 21 respuestas para cerrar el bloque 2 en 60/300. Si se pierde una sesión, el respaldo fijado aquí permite recuperar 39; antes de reanudar desde uno posterior hay que actualizar su URL, SHA-256 y cantidad en la celda de recuperación, tras auditarlo. Los archivos de la sesión temporal no son respaldo persistente.

Código, prompts, preguntas, semillas, controles y orden de condiciones del experimento v4 se mantienen. Solo se agrega una parada operativa después de exportar una respuesta completa.


## 1. Dependencias (reinicia la sesión si ya había otras versiones cargadas)

In [ ]:
import sys, subprocess, importlib.metadata
PINNED = {'transformers':'4.57.6', 'accelerate':'1.11.0', 'bitsandbytes':'0.48.1',
          'sentence-transformers':'5.1.2', 'tokenizers':'0.22.2'}
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                      *[p+'=='+v for p,v in PINNED.items()]])
for module, package in [(p.replace('-', '_'), p) for p in PINNED] + [
    ('numpy','numpy'), ('scipy','scipy'), ('torch','torch'), ('sklearn','scikit-learn'),
    ('huggingface_hub','huggingface-hub')]:
    loaded = sys.modules.get(module)
    if loaded is not None and hasattr(loaded, '__version__'):
        if loaded.__version__ != importlib.metadata.version(package):
            raise RuntimeError('Reinicia la sesión: versión anterior en memoria de ' + package)
print('Dependencias instaladas. No se cargó ningún modelo todavía.')

## 2. Restaurar código, prompts externos y corpus incluidos

In [ ]:
BUNDLE_B64 = (
    '...'
    'OhEg/g//B9AhNbJJ3xAA'
)
EXPECTED = '614e8871bd294af36aec9012e01932ea7128363b5e094a3d72cbd7bb9e614f2a'
FILE_HASHES = {'d3_agentes/__init__.py': 'ee0695533a98ad593db63822f7d3775fe565e1c4a5575f78de54034987d44d9d', 'd3_agentes/bridge.py': '0e15d6153defdb4bc77c0c2a99d32e28e34a31b5f24c97b94fd0e06c23924ca1', 'd3_agentes/common.py': '76f9cb1d436e46f095d37a70ddd9b1ab8c8da0f8e1ce5ee1f8f2f1831077bb20', 'd3_agentes/core.py': '3c52ef07de3c7675168f035144ce5c5fe54d7861bf786b14a8ad527da76dcf44', 'd3_agentes/downloads.py': '9345bffa531e1fb55dcd6d572f8afbc9c324972e29c88103aa297486042924cc', 'd3_agentes/experiment.py': 'd3c6b4b1ed314258e175df95193411782ad63ff39d077fb5b5e723a65c4cc8c8', 'd3_agentes/runtime.py': '112401003b7051b2f462f0496bfa1f6a313a7fa01c7c7a262930837fbc04ab6e', 'd3_agentes/tools.py': 'f075926974535b9b03aa8eaf431b0a803af13d81444b08f136818643a09ddab0', 'prompts/auditor.txt': '69aa2186f17c3a4b86799f4435060ef86865e8be00a37e49f4249bc1df7be9b2', 'prompts/completitud.txt': '72918aa2f8a5bfb7981073823e44145c2574691812672487df6664b540e9c538', 'prompts/descomposicion.txt': 'cf09766bec652248002dfa204b2cf8fff4df6330f22f01c838b1e1c2d4de50f8', 'prompts/investigador.txt': '0bcc8694aa10ecdcfa5c17438071a7390cd77a2a355141191e13b5704ee1c9ea', 'prompts/redactor.txt': '25df34e7d299ddaa25db0f2877dba7965b5129b48ca64e6d92cf7dddb546ef33', 'prompts/verificacion.txt': '4aedfc2bae8ae6a485dacd0f606ff8b8334668fc5915b420481953559a41c1b7', 'd2_runtime/experimento.py': 'd20ce66a54c7791a480e0d9466e1740b34a8c5198df94ec52c1d4868219e22d0', 'd2_runtime/sistema.py': 'be9ff8714f6580b30618ef4d45c98e744463da620e98721701dadfe9521fe130', 'd2_runtime/recuperador_portable.py': 'a04c720b0cf25eb8010996dd28f70f8a08806b6c556e0be6bc2ea3a6fb4dd056', 'data/corpus.json': '0a31ff812a1969ff81bbc895feaba490e3bd027b092e52101296e3c872719400', 'data/fragmentos_busqueda.json': 'c125615c1c3cddb23d69c3b2b93d7a73e076c90fe861cc217b59f293aa9b14cd', 'data/preguntas.json': '89084eb950e1c1bca00c0a90adbf737a500a978cf12fe61bf70f2325ccf15d12', 'data/configuracion.json': 'd5dd7936c24d811b302cc961b926735a8626708a94fc65b21424dad03eaab310'}

import base64, gzip, hashlib, json, sys
from pathlib import Path
packed = base64.b64decode(BUNDLE_B64, validate=True)
if hashlib.sha256(packed).hexdigest() != EXPECTED:
    raise ValueError('Paquete alterado.')
sources = json.loads(gzip.decompress(packed))
if set(sources) != set(FILE_HASHES):
    raise ValueError('Inventario del paquete incorrecto.')
WORKDIR = Path.cwd() / ('d3_nuevo_runtime_' + EXPECTED[:16])
for name, text in sources.items():
    path = Path(name)
    if path.is_absolute() or '..' in path.parts:
        raise ValueError('Ruta no permitida.')
    if hashlib.sha256(text.encode()).hexdigest() != FILE_HASHES[name]:
        raise ValueError('Fuente alterada: ' + name)
    target = WORKDIR / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(text, encoding='utf-8')
previous = sys.modules.get('d3_agentes')
if previous is not None and Path(previous.__file__).resolve().parent != (WORKDIR / 'd3_agentes').resolve():
    raise RuntimeError('Reinicia la sesión para cargar esta versión de D3.')
if str(WORKDIR.resolve()) not in sys.path:
    sys.path.insert(0, str(WORKDIR.resolve()))
del BUNDLE_B64, sources, packed
print('Código y prompts restaurados en', WORKDIR)

## 3. Cargar una sola instancia Qwen y el índice E5

In [ ]:
from d3_agentes.runtime import load, execute
from d3_agentes.common import VARIANTS
from d3_agentes.experiment import export, audit_zip, sealed_load
USAR_DRIVE = False
if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT = Path('/content/drive/MyDrive/GenIA_D3_Agentes/runs')
else:
    OUTPUT = Path.cwd() / 'd3_agentes_runs'
ENV = load(OUTPUT)
print('Preguntas de desarrollo:', len(ENV['questions']))

## 4. Ejecutar un bloque y descargar

Volver a ejecutar esta celda continúa el próximo bloque. No usar Ejecutar todo para continuar en una sesión ya cargada.


In [ ]:
# Ejecutar en Colab después de las secciones 1, 2 y 3. No ejecuta el piloto.
from pathlib import Path, PurePosixPath
import hashlib, json, zipfile, urllib.request
from d3_agentes.experiment import audit_zip, prepare, export
from d3_agentes.common import Limits, PromptBook, sealed_load, VARIANTS

SNAPSHOT_URL = 'https://raw.githubusercontent.com/Pacortes2021/GENERATIVE-ARTIFICIAL-INTELLIGENCE-DELIVERABLE-1/22963a0f4e364e1fbc807883091fe6aebc7d38ca/Deliverable3_Agentes/resultados/lote_99b9b55bd79d4572f743/snapshots/20261028T071051Z_039de300.zip'
SNAPSHOT_SHA256 = 'a9a3b86a82bf2ef1a530ad630ea6c5d8366a3c70d1a19ad6f30bf37d56a4d10d'

def restaurar_respaldo(ENV, OUTPUT):
    backup = Path('/content/respaldo_original_039de300.zip')
    if not backup.exists():
        urllib.request.urlretrieve(SNAPSHOT_URL, backup)
    assert hashlib.sha256(backup.read_bytes()).hexdigest() == SNAPSHOT_SHA256, 'Respaldo distinto.'
    summary = audit_zip(backup)
    assert summary['completadas'] == 39 and summary['esperadas'] == 300
    with zipfile.ZipFile(backup) as archive:
        config = json.loads(archive.read('configuracion.json'))['contenido']
        assert config['inferencia'] == ENV['settings'], 'Entorno distinto: revisar versiones antes de reanudar.'
        assert config['preguntas'] == ENV['questions'] and config['variantes'] == list(VARIANTS)
        RUN_DIR, actual = prepare(ENV['questions'], ENV['retriever'], ENV['settings'], OUTPUT,
                                  tuple(config['variantes']), Limits(**config['limites']),
                                  config['etiqueta'], PromptBook())
        assert actual == config and RUN_DIR.name == 'agentes_99b9b55bd79d4572f743'
        for name in archive.namelist():
            relative = PurePosixPath(name)
            assert not relative.is_absolute() and '..' not in relative.parts
            target = RUN_DIR / name
            if target.exists():
                if name == 'configuracion.json':
                    assert sealed_load(target) == config, 'Configuracion incompatible.'
                elif name.startswith(('resultados/', 'llamadas/', 'fuentes/', 'prompts/')):
                    assert target.read_bytes() == archive.read(name), 'Conflicto con respaldo: ' + name
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(name))
    export(RUN_DIR)
    return RUN_DIR

class BloqueTerminado(BaseException):
    """Parada operativa normal después de exportar una respuesta completa."""


def ejecutar_bloque(ENV, OUTPUT, RUN_DIR, descargar, tamaño=30):
    """Conserva identidad y semillas del lote; ejecuta hasta el siguiente límite."""
    import shutil
    import d3_agentes.experiment as experiment_module
    from d3_agentes.runtime import execute
    original_export = experiment_module.export
    original_export(RUN_DIR)
    inicial = audit_zip(RUN_DIR.with_suffix('.zip'))['completadas']
    if inicial == 300:
        print('Las 300 respuestas ya están conservadas. No se ejecuta inferencia.')
        return RUN_DIR
    assert 0 <= inicial < 300 and tamaño == 30
    objetivo = min((inicial // tamaño + 1) * tamaño, 300)
    numero = objetivo // tamaño
    print(f'BLOQUE {numero}/10: {inicial}/300 conservadas; objetivo {objetivo}/300.', flush=True)
    print(f'Se generarán {objetivo - inicial} respuestas pendientes, sin repetir las guardadas.', flush=True)
    backup_progress = {'n': inicial}

    def export_bloque(folder):
        archive_path = original_export(folder)
        n = json.loads((Path(folder) / 'resumen_tecnico.json').read_text())['completadas']
        # Copias inmutables para que la siguiente exportación no altere una descarga activa.
        if n >= backup_progress['n'] + 6 or n >= objetivo:
            snapshot = Path(folder).parent / (Path(folder).name + f'_respaldo_{n:03}de300.zip')
            shutil.copy2(archive_path, snapshot)
            try:
                descargar(str(snapshot))
                print('DESCARGA SOLICITADA:', n, '/300', snapshot.name, flush=True)
            except Exception as exc:
                print('Descargar desde Archivos:', snapshot, type(exc).__name__, flush=True)
            backup_progress['n'] = n
        if n >= objetivo:
            raise BloqueTerminado()
        return archive_path

    # Solo capa de exportación en memoria. Fuentes, prompts, config y semillas intactos.
    # BaseException evita registrar esta parada prevista como error técnico del modelo.
    experiment_module.export = export_bloque
    try:
        execute(ENV, ENV['questions'], OUTPUT, variants=VARIANTS, tag='lote-v4', verbose=False)
    except BloqueTerminado:
        print(f'BLOQUE {numero}/10 TERMINADO. Verifica y conserva el ZIP antes del siguiente.', flush=True)
    finally:
        experiment_module.export = original_export
        original_export(RUN_DIR)
        progress = audit_zip(RUN_DIR.with_suffix('.zip'))
        print('RESPALDO VERIFICADO:', progress['completadas'], '/', progress['esperadas'], flush=True)
        if progress['completadas'] < objetivo:
            try:
                descargar(str(RUN_DIR.with_suffix('.zip')))
            except Exception as exc:
                print('Descargar respaldo parcial desde Archivos:', type(exc).__name__, flush=True)
    return RUN_DIR


CONTINUAR_HASTA_FINAL = True  # True: ejecuta automáticamente todos los bloques hasta 300; False: solo un bloque de 30


if __name__ == '__main__':
    from google.colab import files
    RUN_DIR = restaurar_respaldo(ENV, OUTPUT)
    if globals().get('CONTINUAR_HASTA_FINAL', False):
        while True:
            RUN_DIR = ejecutar_bloque(ENV, OUTPUT, RUN_DIR, files.download)
            progress = audit_zip(RUN_DIR.with_suffix('.zip'))
            if progress['completadas'] >= 300:
                print('¡ÉXITO TOTAL! 300/300 respuestas generadas y respaldadas.', flush=True)
                break
    else:
        RUN_DIR = ejecutar_bloque(ENV, OUTPUT, RUN_DIR, files.download)
